In [6]:
!git clone https://github.com/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq.git
%cd ml-internship-muhammadahmadishtiaq

Cloning into 'ml-internship-muhammadahmadishtiaq'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 45), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 8.91 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/ml-internship-muhammadahmadishtiaq/ml-internship-muhammadahmadishtiaq


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a **classification** task, not pure ranking or clustering. The lane's own label, `is_declining_label`, is a binary yes/no (page trending down vs not) — that's a classic classification target. The business action then turns the model's predicted probability into a **ranked queue**: the content team doesn't just want a flag, they want to know which pages to check *first*. So the task type is classification underneath, with the output consumed as a ranking (sort pages by predicted decline probability, review the top of the list). It isn't clustering, because we already have a labeled outcome to predict, not an unlabeled grouping problem.

In [7]:
# Section 1 check: confirm the label is binary (classification), not continuous
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_label"].value_counts())
print("Two classes -> confirms binary classification framing.")


is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Two classes -> confirms binary classification framing.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` — 1 if the page is trending down, 0 otherwise.

**Where it comes from — and the honest caveat:** it is *not* an independently observed outcome like "an editor confirmed this page declined" or "traffic actually kept falling next quarter." It's a **defined rule** on measured data: `trend_direction` is computed from `trend_pct`, which compares impressions in the last 30 days vs. the previous 30 days, thresholded at ±20%. So the label is a proxy for "real decline" built from a fixed formula, not a ground-truth business event.

This matters because of the framing skill's rule — *"a label that comes from someone's rule means your model learns the rule, not the world."* Here, `trend_direction` and `trend_pct` are therefore **never allowed as model features** (confirmed in the data dictionary) — using them would let the model just re-derive the label instead of learning a real pattern. The model has to predict the same outcome using *other* signals (position, CTR, content age, word count, keyword context, etc.), which is what makes this a genuine prediction problem rather than a lookup.

In [8]:
# Section 2 check: confirm the label-source columns are excluded from candidate features
forbidden = {"trend_direction", "trend_pct"}
candidate_features = [col for col in df.columns if col not in forbidden and col != "is_declining_label"]
print("Forbidden as features (label source):", forbidden)
print("Number of usable candidate columns:", len(candidate_features))


Forbidden as features (label source): {'trend_direction', 'trend_pct'}
Number of usable candidate columns: 42


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: precision@K** (precision among the top-K pages the model flags for review).

Why this one over plain accuracy: the action is a content team working through a **ranked queue**, reviewing a limited number of pages per week — not labeling every page in the dataset. What matters is: *of the top pages we send to review, how many are actually declining?* That's precision at the top of the ranking, not overall accuracy across all 30,000 pages. A high accuracy score is easy here anyway, since the classes are close to balanced (54% declining) — precision@K is the number that maps directly to "was the reviewer's time well spent."

**Secondary/sanity metric: ROC-AUC**, to confirm the model separates the two classes better than chance before trusting the ranking at all.

**Good = beats the baseline.** "Good" isn't an absolute number — it's precision@K measurably above the base decline rate (~54%, computed below) and above a simple non-ML baseline. Naming the metric now, before any model exists, is what keeps this honest later.

In [9]:
# Section 3 check: the baseline rate any metric has to beat
base_rate = df["is_declining_label"].mean()
print(f"Base decline rate: {base_rate:.1%}")
print("A model's precision@K only means something if it clears this baseline by a real margin.")


Base decline rate: 54.2%
A model's precision@K only means something if it clears this baseline by a real margin.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (one page).** `content_id` is the unique pseudonymized key; `client_id` groups pages by client (32 clients in this slice) — used only for grouped train/test splits, never as a feature. Loaded straight from the starter dataset named in `skills/flyrank/flyrank-data/SKILL.md`.

In [10]:
# Section 4: show the real unit of analysis
print("Shape:", df.shape)
print("Unique content_id (should equal row count):", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())

# One row = one page. Show a few real rows including the target column.
cols_to_show = ["content_id", "client_id", "content_type", "word_count",
                "avg_position", "ctr", "impressions_90d", "is_declining_label"]
df[cols_to_show].head(5)


Shape: (30000, 45)
Unique content_id (should equal row count): 30000
Unique clients: 32


,content_id,client_id,content_type,word_count,avg_position,ctr,impressions_90d,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,10.6,0.76,3803,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,20.3,0.05,15320,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,36.5,0.09,12581,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,6.2,0.49,11751,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,44.0,0.13,19140,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement already exists in this data — it's literally how the label itself was built (a threshold on 30-day impression change). That rule only looks at one signal in isolation. Whether a page is *actually* worth an editor's time depends on many signals moving together: search position, CTR, content age, freshness, word count, keyword competition, content type, AI-referred traffic share — and how those interact (e.g. a page with falling impressions but excellent position and rising engagement is a very different case than one falling on every dimension at once). Hand-writing an if/else that weighs 40+ correlated, partly-missing signals correctly — and re-weighs them as patterns shift client to client and month to month — isn't realistic. That's exactly the situation the framing skill flags as "real but too messy to write by hand": ML can learn which combinations of signals matter and by how much, and can be re-validated as the patterns drift, instead of freezing one person's guess about which threshold matters most.

In [11]:
# Section 5 check: a taste of why one signal alone isn't enough --
# pages that are 'declining' by the label still span very different situations.
declining = df[df["is_declining_label"] == 1]
print("Declining pages -- spread across position tiers (not one clean pattern):")
print(declining["position_tier"].value_counts(normalize=True).round(2))


Declining pages -- spread across position tiers (not one clean pattern):
position_tier
page_1      0.41
striking    0.27
page_3_5    0.25
top_3       0.03
deep        0.03
Name: proportion, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.